# Projeto Lakehouse com Delta Lake

## Objetivo
Construir um pipeline de dados em arquitetura Lakehouse utilizando PySpark e Delta Lake, com base em dados do Campeonato Brasileiro.

In [1]:
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip

builder = SparkSession.builder \
    .appName("Projeto Lakehouse Brasileirão") \
    .master("local[*]") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")

spark = configure_spark_with_delta_pip(builder).getOrCreate()

print("Spark + Delta iniciados com sucesso!")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/04/28 20:02:11 WARN Utils: Your hostname, MacBook-Air-de-Raul.local, resolves to a loopback address: 127.0.0.1; using 10.1.1.217 instead (on interface en0)
26/04/28 20:02:11 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
:: loading settings :: url = jar:file:/Users/castroderaul/Library/Caches/pypoetry/virtualenvs/projeto-lakehouse-JYffaGe1-py3.11/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /Users/castroderaul/.ivy2.5.2/cache
The jars for the packages stored in: /Users/castroderaul/.ivy2.5.2/jars
io.delta#delta-spark_4.1_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-cf50523f-7185-4364-8520-09fb980aeb48;1.0
	confs: [default]
	found io.delta#delta-spark_4.1_2.13;4.2.0 in central
	found io.delta#delta-storage;4.2.0 in central
	found io.unitycatalog#unityca

Spark + Delta iniciados com sucesso!


## Fonte de Dados (Raw)

Foi utilizado um arquivo CSV contendo informações históricas do Brasileirão, armazenado na camada raw.

In [2]:
df_raw = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .csv("../data/raw/brasileirao.csv")

df_raw.show(5)

+--------------+----------+------+--------------------+-------+-------+-----------+-------------+--------------+----------------+-----------------+------------------+-------------------+-----------------------------+------------------------------+----------------------------+-----------------------------+-------------+--------------+---------------------+----------------------+-------------------+--------------------+---------------+----------------+---------------------------+----------------------------+----------------+-----------------+---------------------+----------------------+---------------+----------------+--------------------+---------------------+
|ano_campeonato|      data|rodada|             estadio|arbitro|publico|publico_max|time_mandante|time_visitante|tecnico_mandante|tecnico_visitante|colocacao_mandante|colocacao_visitante|valor_equipe_titular_mandante|valor_equipe_titular_visitante|idade_media_titular_mandante|idade_media_titular_visitante|gols_mandante|gols_visitan

26/04/28 20:02:18 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


In [3]:
df_raw.printSchema()

root
 |-- ano_campeonato: integer (nullable = true)
 |-- data: date (nullable = true)
 |-- rodada: integer (nullable = true)
 |-- estadio: string (nullable = true)
 |-- arbitro: string (nullable = true)
 |-- publico: integer (nullable = true)
 |-- publico_max: integer (nullable = true)
 |-- time_mandante: string (nullable = true)
 |-- time_visitante: string (nullable = true)
 |-- tecnico_mandante: string (nullable = true)
 |-- tecnico_visitante: string (nullable = true)
 |-- colocacao_mandante: integer (nullable = true)
 |-- colocacao_visitante: integer (nullable = true)
 |-- valor_equipe_titular_mandante: integer (nullable = true)
 |-- valor_equipe_titular_visitante: integer (nullable = true)
 |-- idade_media_titular_mandante: double (nullable = true)
 |-- idade_media_titular_visitante: double (nullable = true)
 |-- gols_mandante: integer (nullable = true)
 |-- gols_visitante: integer (nullable = true)
 |-- gols_1_tempo_mandante: integer (nullable = true)
 |-- gols_1_tempo_visitante: 

In [4]:
df_raw.write \
    .format("delta") \
    .mode("overwrite") \
    .save("../data/bronze/brasileirao")

## Camada Bronze

Nesta etapa, os dados brutos foram convertidos para o formato Delta Lake, preservando o conteúdo original para consultas futuras.

In [5]:
df_bronze = spark.read \
    .format("delta") \
    .load("../data/bronze/brasileirao")

df_bronze.show(5)

+--------------+----------+------+--------------------+-------+-------+-----------+-------------+--------------+----------------+-----------------+------------------+-------------------+-----------------------------+------------------------------+----------------------------+-----------------------------+-------------+--------------+---------------------+----------------------+-------------------+--------------------+---------------+----------------+---------------------------+----------------------------+----------------+-----------------+---------------------+----------------------+---------------+----------------+--------------------+---------------------+
|ano_campeonato|      data|rodada|             estadio|arbitro|publico|publico_max|time_mandante|time_visitante|tecnico_mandante|tecnico_visitante|colocacao_mandante|colocacao_visitante|valor_equipe_titular_mandante|valor_equipe_titular_visitante|idade_media_titular_mandante|idade_media_titular_visitante|gols_mandante|gols_visitan

## Camada Silver

Na camada Silver, foram selecionadas colunas relevantes e aplicados tratamentos básicos para melhorar a qualidade dos dados.

In [6]:
from pyspark.sql.functions import col

df_silver = df_bronze.select(
    "ano_campeonato",
    "data",
    "rodada",
    "time_mandante",
    "time_visitante",
    "gols_mandante",
    "gols_visitante",
    "publico"
).dropna(subset=["time_mandante", "time_visitante"])

df_silver.show(5)

+--------------+----------+------+-------------+--------------+-------------+--------------+-------+
|ano_campeonato|      data|rodada|time_mandante|time_visitante|gols_mandante|gols_visitante|publico|
+--------------+----------+------+-------------+--------------+-------------+--------------+-------+
|          2005|2005-10-08|    31|       Paraná|    Fluminense|            6|             1|   NULL|
|          2005|2005-04-23|     1|       Paraná|      Goiás EC|            0|             2|   NULL|
|          2005|2005-08-21|    21|       Paraná| Vasco da Gama|            0|             0|   NULL|
|          2005|2005-08-28|    23|       Paraná|     São Paulo|            0|             4|   NULL|
|          2005|2005-06-26|     9|     Botafogo|Figueirense FC|            0|             1|   NULL|
+--------------+----------+------+-------------+--------------+-------------+--------------+-------+
only showing top 5 rows


## Camada Gold

Foi construída uma análise com os 5 maiores públicos registrados no Estádio Heriberto Hülse de 2005 a 2024 no Campeonato Brasileiro.

In [7]:
from pyspark.sql.functions import col

df_gold = df_silver.filter(
    col("estadio") == "Estádio Heriberto Hülse"
).select(
    "data",
    "time_mandante",
    "time_visitante",
    "publico"
).orderBy(
    col("publico").desc()
).limit(5)

df_gold.show(truncate=False)

+----------+-------------+--------------+-------+
|data      |time_mandante|time_visitante|publico|
+----------+-------------+--------------+-------+
|2024-06-02|Criciúma EC  |Palmeiras     |19027  |
|2024-08-18|Criciúma EC  |Vasco da Gama |18403  |
|2013-12-01|Criciúma EC  |São Paulo     |18081  |
|2024-06-14|Internacional|São Paulo     |17987  |
|2013-06-08|Criciúma EC  |Flamengo      |17785  |
+----------+-------------+--------------+-------+



In [8]:
df_gold.write \
    .format("delta") \
    .mode("overwrite") \
    .save("../data/gold/top5_publicos_heriberto_hulse")

## Conclusão

O projeto demonstrou a construção de um pipeline Lakehouse utilizando Delta Lake, com separação em camadas Raw, Bronze, Silver e Gold, permitindo armazenamento confiável e geração de insights analíticos.